# FEMI-3A | Fixed-provider will/may replication
This notebook has **no live calls unless you run the marked LIVE cell**. Contains 24 newly executed ML cases, 288 planned OpenRouter calls and a three-wording × two-evidence factorial. The record-present arm already supplies an authoritative verdict. The record-absent arm must DEFER.

**Security:** revoke the API key previously exposed in chat. Save the *new* key in Colab Secrets as `OPENROUTER_API_KEY`. Never paste or print it in a cell. No key will be saved in the output archive.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, os, shutil
uploaded=files.upload()  # select the entire FEMI3A_MinimalWording_20261108.zip
archive=Path(next(iter(uploaded)))
assert archive.suffix=='.zip'
with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
        # Extract only contents within the expected top-level directory.
        assert member.startswith('FEMI3A_MinimalWording_20261108/')
        assert '..' not in Path(member).parts and not member.startswith('/')
    z.extractall('/content')
P=Path('/content/FEMI3A_MinimalWording_20261108')
assert (P/'offline/freeze.json').exists()
print('Loaded frozen FEMI-3A packet:',P)


## Offline preflight (NO paid API calls)
The execution ledger and frozen request manifest have already been built; this cell checks their integrity and reruns the local sklearn evidence generation for consistency.


In [ ]:
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],cwd=P,check=True)
subprocess.run(['python','run_femi3a.py'],cwd=P,check=True)


## LIVE: 288 paid model calls (explicitly opt in)
This experiment is pinned to **Groq**, model `openai/gpt-oss-120b`, with fallbacks disabled. The collection will stop if the provider differs or isn't available; do not change the frozen prompts to force it through another provider. No API credentials are printed. If interrupted, rerun the live cell after restoring any saved checkpoint ZIP.


In [ ]:
from google.colab import userdata
new_key=userdata.get('OPENROUTER_API_KEY')
assert new_key and new_key.strip(), 'Set a NEW OPENROUTER_API_KEY under Colab Secrets'
env=os.environ.copy();env['OPENROUTER_API_KEY']=new_key
try:
    subprocess.run(['python','run_femi3a.py','--live','--out','femi3a_results'],cwd=P,env=env,check=True)
finally:
    env.pop('OPENROUTER_API_KEY',None)
    del new_key


## Optional checkpoint download
If you need to preserve completed responses before a Colab runtime reset, run this cell. To restore after resetting, extract the checkpoint ZIP under the experiment directory before rerunning the live cell. It does **not** contain the API key.


In [ ]:
results=P/'femi3a_results'
if results.exists():
    checkpoint=shutil.make_archive('/content/FEMI3A_CHECKPOINT','zip',root_dir=P,base_dir='femi3a_results')
    files.download(checkpoint)
else:
    print('No saved results yet.')


## Analyze only when all 288 responses are present
The analysis is strict: any missing or duplicate IDs, request hash mismatch, stored parsing inconsistency or provider drift stops the analysis.


In [ ]:
subprocess.run(['python','analyze_femi3a.py','--results','femi3a_results/responses.jsonl','--out','femi3a_results/analysis'],cwd=P,check=True)
output=shutil.make_archive('/content/FEMI3A_LIVE_RESULTS','zip',root_dir=P/'femi3a_results')
files.download(output)
